In [1]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.3 MB 3.5 MB/s eta 0:00:05
   - -------------------------------------- 0.8/16.3 MB 3.8 MB/s eta 0:00:05
   ---- ----------------------------------- 1.8/16.3 MB 3.4 MB/s eta 0:00:05
   ------- -------------------------------- 3.1/16.3 MB 4.1 MB/s eta 0:00:04
   --------- ------------------------------ 3.9/16.3 MB 4.1 MB/s eta 0:00:04
   ------------ --------------------------- 5.0/16.3 MB 4.3 MB/s eta 0:00:03
   ------------- -------------------------- 5.5/16.3 MB 4.0 MB/s eta 0:00:03
   -------------- ------------------------- 6.0/16.3 MB 3.9 MB/s eta 0:00:03
   -------------- ------------------------- 6.0/16.3 MB 3.9 MB/s eta 0:00:03
   ---------------- ----------------------- 6.6/16.3 MB 3.3 MB/s eta 0:00:03
   ----------------- ---------------------- 7.1/16.3 MB 3.3 MB/s eta 0:00:03
   ------------------ --------------------- 7.6/16.3 MB 3.2 MB/s eta 0:00:03
   ---


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import numpy as np
import pandas as pd
import faiss

In [3]:
project_dir = os.path.join(
    os.path.expanduser("~"),
    "AI_Software_Engineering_Agent"
)

processed_dir = os.path.join(
    project_dir,
    "data",
    "processed_code"
)

os.makedirs(processed_dir, exist_ok=True)

print("Processed directory:")
print(processed_dir)

Processed directory:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\processed_code


In [4]:
#Loading phase4 embeddings
embeddings_path = os.path.join(
    processed_dir,
    "embeddings.npy"
)

embeddings = np.load(embeddings_path)

print("Embedding shape:", embeddings.shape)

Embedding shape: (30650, 384)


In [5]:
assert embeddings.shape[0] == 30650
print("✓ All 30,650 embeddings loaded successfully.")

✓ All 30,650 embeddings loaded successfully.


In [6]:
#Loading phase2+3 ouputs
chunks_path = os.path.join(
    processed_dir,
    "code_chunks.json"
)

with open(chunks_path, "r", encoding="utf-8") as f:
    code_chunks = json.load(f)

print("Total code chunks:", len(code_chunks))

Total code chunks: 30650


In [7]:
assert len(code_chunks) == len(embeddings)

print("✓ Number of chunks matches number of embeddings.")

✓ Number of chunks matches number of embeddings.


In [8]:
#Convert embeddings to FAISS-compatible format
embeddings = embeddings.astype("float32")

print("Data type:", embeddings.dtype)

Data type: float32


In [9]:
#Normalize the vectors
faiss.normalize_L2(embeddings)

print("✓ Embeddings normalized successfully.")

✓ Embeddings normalized successfully.


In [10]:
#Creating the FAISS index
dimension = embeddings.shape[1]

print("Embedding dimension:", dimension)

Embedding dimension: 384


In [11]:
index = faiss.IndexFlatIP(dimension)

print("✓ FAISS index created.")

✓ FAISS index created.


In [12]:
#Add all embeddings to FAISS
index.add(embeddings)

print("Total vectors in FAISS:", index.ntotal)

Total vectors in FAISS: 30650


In [13]:
#Saving the FAISS index
faiss_index_path = os.path.join(
    processed_dir,
    "faiss_index.bin"
)

faiss.write_index(
    index,
    faiss_index_path
)

print("FAISS index saved successfully.")
print(faiss_index_path)

FAISS index saved successfully.
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\processed_code\faiss_index.bin


In [14]:
#Creating vector metadata
metadata = []

for i, chunk in enumerate(code_chunks):

    metadata.append({
        "vector_id": i,
        "chunk_id": chunk.get("chunk_id"),
        "file_id": chunk.get("file_id"),
        "repository": chunk.get("repository"),
        "repository_url": chunk.get("repository_url"),
        "file_name": chunk.get("file_name"),
        "file_path": chunk.get("file_path"),
        "language": chunk.get("language"),
        "chunk_type": chunk.get("chunk_type"),
        "name": chunk.get("name"),
        "parent_class": chunk.get("parent_class"),
        "line_start": chunk.get("line_start"),
        "line_end": chunk.get("line_end")
    })

In [15]:
metadata_df = pd.DataFrame(metadata)

print("Metadata records:", len(metadata_df))

metadata_df.head()

Metadata records: 30650


,vector_id,chunk_id,file_id,repository,repository_url,file_name,file_path,language,chunk_type,name,parent_class,line_start,line_end
0,0,requests_000001,3,requests,https://github.com/psf/requests.git,flask_theme_support.py,docs\_themes\flask_theme_support.py,Python,class,FlaskyStyle,None,7,86
1,1,requests_000002,4,requests,https://github.com/psf/requests.git,adapters.py,src\requests\adapters.py,Python,function,_urllib3_request_context,None,85,119
2,2,requests_000003,4,requests,https://github.com/psf/requests.git,adapters.py,src\requests\adapters.py,Python,class,BaseAdapter,None,122,155
3,3,requests_000004,4,requests,https://github.com/psf/requests.git,adapters.py,src\requests\adapters.py,Python,method,__init__,BaseAdapter,125,126
4,4,requests_000005,4,requests,https://github.com/psf/requests.git,adapters.py,src\requests\adapters.py,Python,method,send,BaseAdapter,128,151


In [16]:
#Saving the metadata
metadata_path = os.path.join(
    processed_dir,
    "vector_metadata.json"
)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print("✓ Vector metadata saved.")

✓ Vector metadata saved.


In [17]:
#Loading the FAISS index again
loaded_index = faiss.read_index(
    faiss_index_path
)

print("Loaded FAISS vectors:", loaded_index.ntotal)

Loaded FAISS vectors: 30650


In [18]:
#Testing semantic search
query_vector = embeddings[0].reshape(1, -1)

In [19]:
k = 5

distances, indices = loaded_index.search(
    query_vector,
    k
)

print("Vector IDs:")
print(indices)

print("\nSimilarity scores:")
print(distances)

Vector IDs:
[[   0  921  817  804 1195]]

Similarity scores:
[[1.         0.65449095 0.64755726 0.63611716 0.63007927]]


In [20]:
#Displaying the retrieved code
for rank, vector_id in enumerate(indices[0]):

    chunk = code_chunks[vector_id]

    print("\n" + "=" * 80)
    print(f"Rank: {rank + 1}")
    print(f"Similarity: {distances[0][rank]:.4f}")
    print(f"Repository: {chunk.get('repository')}")
    print(f"File: {chunk.get('file_name')}")
    print(f"Path: {chunk.get('file_path')}")
    print(f"Type: {chunk.get('chunk_type')}")
    print(f"Name: {chunk.get('name')}")
    print("=" * 80)


Rank: 1
Similarity: 1.0000
Repository: requests
File: flask_theme_support.py
Path: docs\_themes\flask_theme_support.py
Type: class
Name: FlaskyStyle

Rank: 2
Similarity: 0.6545
Repository: flask
File: helpers.py
Path: src\flask\helpers.py
Type: method
Name: __init__

Rank: 3
Similarity: 0.6476
Repository: flask
File: cli.py
Path: src\flask\cli.py
Type: function
Name: get_version

Rank: 4
Similarity: 0.6361
Repository: flask
File: blueprints.py
Path: src\flask\blueprints.py
Type: class
Name: Blueprint

Rank: 5
Similarity: 0.6301
Repository: flask
File: test_async.py
Path: tests\test_async.py
Type: class
Name: AppError


In [21]:
#Creating a usable search function
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
def search_code(query, top_k=5):

    # Convert query to embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query vector
    faiss.normalize_L2(query_embedding)

    # Search FAISS
    distances, indices = loaded_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, vector_id in enumerate(indices[0]):

        chunk = code_chunks[vector_id]

        results.append({
            "rank": rank + 1,
            "score": float(distances[0][rank]),
            "repository": chunk.get("repository"),
            "file_name": chunk.get("file_name"),
            "file_path": chunk.get("file_path"),
            "chunk_type": chunk.get("chunk_type"),
            "name": chunk.get("name"),
            "code": chunk.get("code")
        })

    return results

In [23]:
#Test with a real developer question
results = search_code(
    "How are HTTP requests handled?",
    top_k=5
)

In [24]:
for result in results:

    print("\n" + "=" * 80)
    print("Rank:", result["rank"])
    print("Score:", round(result["score"], 4))
    print("Repository:", result["repository"])
    print("File:", result["file_name"])
    print("Path:", result["file_path"])
    print("Type:", result["chunk_type"])
    print("Name:", result["name"])


Rank: 1
Score: 0.4917
Repository: httpx
File: _client.py
Path: httpx\_client.py
Type: class
Name: Client

Rank: 2
Score: 0.4688
Repository: httpx
File: base.py
Path: httpx\_transports\base.py
Type: method
Name: handle_request

Rank: 3
Score: 0.4621
Repository: httpx
File: _client.py
Path: httpx\_client.py
Type: class
Name: AsyncClient

Rank: 4
Score: 0.4561
Repository: httpx
File: base.py
Path: httpx\_transports\base.py
Type: class
Name: BaseTransport

Rank: 5
Score: 0.456
Repository: httpx
File: _main.py
Path: httpx\_main.py
Type: function
Name: main
